# 🗄️ Day 148 — FastAPI + SQLite Database Integration & API Authentication
## Month 8 | Streamlit + FastAPI Deployment | Deepanshu Garg

---

| Field | Details |
|---|---|
| **Month** | 8 — Streamlit + FastAPI Deployment |
| **Day** | 148 (Month 8, Week 2, Day 3) |
| **Topic** | FastAPI + SQLAlchemy + SQLite · CRUD endpoints · API key authentication · Background tasks |
| **Dataset** | FreelanceHub India (500 rows, seed=141) |
| **Deliverables** | `day148_api/main.py` · `day148_api/database.py` · `day148_api/models.py` · `day148_api/auth.py` · `test_day148.py` |
| **Max Score** | 80 pts + 10★ Bonus |
| **GitHub Repo** | Month8-Streamlit-FastAPI-Portfolio |

---

## 🗺️ Month 8 Scorecard

| Day | Topic | Score |
|---|---|---|
| 141 | Streamlit Fundamentals | 80/80 + 10★ ✅ |
| 142 | Session State + Tabs + Forms | 80/80 + 10★ ✅ |
| 143 | File Upload + Dynamic EDA | 80/80 + 10★ ✅ |
| 144 | ML Model Deployment (Basic) | 80/80 + 10★ ✅ |
| 145 | Advanced Caching + Multi-Page Apps | 80/80 + 10★ ✅ |
| 146 | ML Explainability Dashboard (SHAP) | 80/80 + 10★ ✅ |
| 147 | FastAPI Fundamentals | ← submit or ✅ |
| **148** | **FastAPI + DB + Auth** | **← Today** |

---

## 🔄 Why Database + Auth Today?

```
Day 147 (Fundamentals)              Day 148 (Production-grade)
────────────────────────────        ─────────────────────────────────────────
API accepts requests ✅             API persists data to SQLite ✅
Returns JSON predictions ✅         API keys restrict who can call it ✅
No memory between calls             Queries filter, paginate, aggregate
Anyone can call any endpoint        Only authorised clients get results

The gap between "demo" and "client-ready API":
  1. Persistence  — SQLite stores every project + prediction log
  2. Auth         — API keys prevent unauthorized access
  3. CRUD         — CREATE / READ / FILTER / DELETE via endpoints
```

> **Freelance angle:** A client paying ₹25k/month for your ML API  
> expects two things before going live: (1) data doesn't vanish on restart,  
> (2) competitors can't call their endpoint for free. Days 147 + 148 = that gap closed.


---
## 📂 Section 1 — Raw Data
> ⚠️ **NEVER MODIFY THIS SECTION.** Read-only reference.


In [1]:
import numpy as np
import pandas as pd

np.random.seed(141)
n = 500
categories      = ['Web Dev', 'Data Science', 'Graphic Design', 'Content Writing', 'SEO']
experience_lvls = ['Junior', 'Mid', 'Senior']
payment_types   = ['Fixed', 'Hourly']

df = pd.DataFrame({
    'project_id':       range(1, n+1),
    'category':         np.random.choice(categories, n),
    'experience_level': np.random.choice(experience_lvls, n),
    'hourly_rate':      np.round(np.random.uniform(5, 80, n), 2),
    'client_rating':    np.where(np.random.rand(n) < 0.08, np.nan,
                            np.round(np.random.uniform(1, 5, n), 1)),
    'bids_received':    np.where(np.random.rand(n) < 0.06, np.nan,
                            np.random.randint(1, 60, n).astype(float)),
    'payment_type':     np.random.choice(payment_types, n),
    'duration_days':    np.where(np.random.rand(n) < 0.05, np.nan,
                            np.random.randint(3, 120, n).astype(float)),
    'status':           np.random.choice(['Completed', 'Cancelled', 'In Progress'],
                            n, p=[0.55, 0.25, 0.20]),
    'platform':         np.random.choice(['Upwork', 'Fiverr', 'Freelancer'], n),
})

# Fill NaN
df['client_rating'] = df['client_rating'].fillna(df['client_rating'].median())
df['bids_received'] = df['bids_received'].fillna(df['bids_received'].median())
df['duration_days'] = df['duration_days'].fillna(df['duration_days'].median())

print("Shape:", df.shape)
print(df.head(3).to_string())
print("\nStatus counts:\n", df['status'].value_counts().to_string())


Shape: (500, 10)
   project_id         category experience_level  hourly_rate  client_rating  bids_received payment_type  duration_days     status platform
0           1     Data Science              Mid        70.45            3.0           32.0       Hourly           82.0  Cancelled   Fiverr
1           2     Data Science              Mid        34.26            3.0           51.0        Fixed           47.0  Cancelled   Upwork
2           3  Content Writing           Senior        75.32            2.1           32.0       Hourly           53.0  Cancelled   Upwork

Status counts:
 status
Completed      269
Cancelled      141
In Progress     90


---
## 📖 Section 2 — Concept Notes

### 2.1 SQLAlchemy ORM in FastAPI

**SQLAlchemy** is Python's most-used database toolkit. In FastAPI, it handles:

```
Your Python class  →  SQLAlchemy ORM  →  SQL table
Project(id=1, ...)       maps to        projects table row 1
```

**Three-file pattern (industry standard):**

| File | Responsibility |
|---|---|
| `database.py` | Engine, session factory, Base class |
| `models.py` | Table definitions as Python classes |
| `main.py` | Endpoints that create/query sessions |

### 2.2 Session lifecycle — the critical pattern

```python
# WRONG — session never closed on error
db = SessionLocal()
result = db.query(Project).all()
return result

# RIGHT — dependency injection auto-closes session
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

@app.get("/projects")
def read_projects(db: Session = Depends(get_db)):
    return db.query(Project).all()
```
> `Depends(get_db)` is FastAPI's dependency injection — it creates a session  
> per request and closes it automatically, even if the endpoint raises an error.

### 2.3 API Key Authentication

```
Client Request
  → Header: X-API-Key: freelancehub-2024-secret
      ↓
  Dependency: verify_api_key()
      ↓
  Valid?  → YES → endpoint executes
           NO  → 401 Unauthorized returned
```

**Why not JWT here?** JWTs are for user login sessions (stateful, expire, carry claims).  
API keys are for service-to-service calls (simpler, long-lived, per-client).  
Rule of thumb: users log in with JWT, machines authenticate with API keys.

### 2.4 CRUD mapping to HTTP methods

| Operation | HTTP Method | SQLAlchemy | Use case |
|---|---|---|---|
| CREATE | POST | `db.add(); db.commit()` | Insert new project |
| READ (all) | GET | `db.query(Model).all()` | List projects |
| READ (filter) | GET | `.filter(Model.col == val)` | Filter by status |
| READ (aggregate) | GET | `func.avg()` | Stats endpoint |
| DELETE | DELETE | `db.delete(obj); db.commit()` | Remove record |

### 2.5 Background Tasks

FastAPI's `BackgroundTasks` lets you log, send email, or write to DB  
**after** the response has already been returned to the client:

```python
@app.post("/predict")
def predict(data: Input, background_tasks: BackgroundTasks, db = Depends(get_db)):
    result = model.predict(data)
    background_tasks.add_task(log_prediction, db, data, result)  # runs AFTER response
    return result
```
> Client gets their prediction in <50ms. DB logging happens in the background.  
> This is production-grade — never make the client wait for your housekeeping.


---
## ✍️ Section 3 — Practice Tasks

**Folder structure to create:**
```
day148_api/
    __init__.py          (empty)
    database.py          (Task 1)
    models.py            (Task 1)
    auth.py              (Task 2)
    main.py              (Tasks 3–5)
test_day148.py           (Task 6)
```

---

### Task 1 — Database Setup: `database.py` + `models.py` (15 pts)

**`day148_api/database.py`** — engine + session factory:
```python
from sqlalchemy import create_engine
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

DATABASE_URL = "sqlite:///./freelancehub.db"

engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()
```

**`day148_api/models.py`** — two ORM tables:

```python
from sqlalchemy import Column, Integer, String, Float, DateTime
from sqlalchemy.sql import func
from .database import Base

class Project(Base):
    __tablename__ = "projects"
    id              = Column(Integer, primary_key=True, index=True)
    project_id      = Column(Integer, unique=True, index=True)
    category        = Column(String)
    experience_level= Column(String)
    hourly_rate     = Column(Float)
    client_rating   = Column(Float)
    bids_received   = Column(Float)
    payment_type    = Column(String)
    duration_days   = Column(Float)
    status          = Column(String, index=True)
    platform        = Column(String)

class PredictionLog(Base):
    __tablename__ = "prediction_logs"
    id              = Column(Integer, primary_key=True, index=True)
    project_id      = Column(Integer)
    predicted_status= Column(String)
    confidence      = Column(Float)
    created_at      = Column(DateTime(timezone=True), server_default=func.now())
```

**Verify:** Run `python -c "from day148_api.models import Project, PredictionLog; print('Models OK')"` — should print Models OK.

---

### Task 2 — API Key Auth: `auth.py` (10 pts)

**`day148_api/auth.py`**:
```python
from fastapi import Header, HTTPException, status

VALID_API_KEYS = {
    "freelancehub-2024-secret": "admin",
    "client-readonly-key-001":  "readonly",
}

def verify_api_key(x_api_key: str = Header(...)):
    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid or missing API key",
            headers={"WWW-Authenticate": "APIKey"},
        )
    return VALID_API_KEYS[x_api_key]  # returns role: "admin" or "readonly"
```

**Test it manually (in a cell below):**
```python
# Simulate what FastAPI does — call verify_api_key with a bad key
from day148_api.auth import verify_api_key, VALID_API_KEYS
print("Valid keys:", list(VALID_API_KEYS.keys()))
print("admin role?", VALID_API_KEYS.get("freelancehub-2024-secret"))
print("bad key?",    VALID_API_KEYS.get("hacker-key-xyz"))  # should be None
```

---

### Task 3 — Seed Endpoint + CRUD READ endpoints in `main.py` (20 pts)

Build `day148_api/main.py` with the following endpoints:

| Endpoint | Method | Auth? | Returns |
|---|---|---|---|
| `GET /` | GET | No | Status message + endpoint list |
| `POST /seed-db` | POST | Yes (admin) | Seed all 500 rows into `projects` table |
| `GET /projects` | GET | Yes | All projects (paginated: `skip=0, limit=50`) |
| `GET /projects/stats` | GET | Yes | Aggregated stats |
| `GET /projects/filter` | GET | Yes | Filter by `status=` query param |
| `DELETE /projects/{project_id}` | DELETE | Yes (admin) | Delete one row |

**Imports + app setup:**
```python
from fastapi import FastAPI, Depends, HTTPException, Query, BackgroundTasks
from sqlalchemy.orm import Session
from sqlalchemy import func
import numpy as np, pandas as pd, joblib
from . import models
from .database import engine, get_db
from .auth import verify_api_key

models.Base.metadata.create_all(bind=engine)
app = FastAPI(title="FreelanceHub API v2 — DB Edition", version="2.0.0")
```

**Seed helper (call this inside `POST /seed-db`):**
```python
def _build_dataset():
    np.random.seed(141)
    n = 500
    # ... (same generation code as Section 1 — paste it here)
    return df   # returns the cleaned DataFrame
```

**Stats endpoint must return:**
```python
{
  "total_projects": <int>,
  "completed": <int>,
  "cancelled": <int>,
  "in_progress": <int>,
  "avg_hourly_rate": <float, 2 dp>,
  "high_value_count": <int>,    # hourly_rate > 50
  "top_platform": <str>
}
```

**Filter endpoint:** `GET /projects/filter?status=Cancelled` → returns list of matching projects.

---

### Task 4 — Background Task: Log Predictions (15 pts)

Re-use the trained model from Day 147 (or re-train inline).  
Add `POST /predict` to `main.py` that:

1. Accepts the same `ProjectInput` Pydantic model as Day 147.
2. Makes a prediction (same encoding logic as Day 147).
3. **Before returning**, registers a background task: `log_to_db(db, project_id, predicted_status, confidence)`.
4. Returns the prediction JSON immediately (client doesn't wait for logging).

The `log_to_db` function inserts a row into `prediction_logs`.

**Background task function:**
```python
def log_to_db(db: Session, project_id: int, predicted: str, confidence: float):
    entry = models.PredictionLog(
        project_id=project_id,
        predicted_status=predicted,
        confidence=round(confidence, 4)
    )
    db.add(entry)
    db.commit()
```

**`POST /predictions/log`** → return all rows from `prediction_logs` (requires auth).

---

### Task 5 — DELETE endpoint (admin only) (10 pts)

```
DELETE /projects/{project_id}
```
- Auth required: only the `"admin"` role (check the role returned by `verify_api_key`).
- If `project_id` not found → raise `404 Not Found`.
- If role is `"readonly"` → raise `403 Forbidden`.
- On success → return `{"deleted": project_id, "status": "removed"}`.

---

### Task 6 — `test_day148.py` (10 pts)

Write a test script using `httpx` (sync client) that:

1. Tests `GET /` — no auth needed, status 200.
2. Tests `POST /seed-db` with valid key → expect `{"seeded": 500}`.
3. Tests `GET /projects/stats` — verify `total_projects == 500`, `completed == 269`.
4. Tests `GET /projects/filter?status=Cancelled` — verify count == 141.
5. Tests `GET /projects` with **invalid** key — verify 401 response.
6. (Bonus) Tests `DELETE /projects/1` with `readonly` key → verify 403.

Run against `uvicorn day148_api.main:app --reload` (start the server first).

---

### ★ Bonus Task — `/analytics` endpoint (10★ pts)

Add `GET /analytics` that returns a per-category breakdown:
```python
{
  "by_category": [
    {"category": "Web Dev",        "count": 113, "avg_rate": 41.46, "completed_pct": 54.9},
    {"category": "Data Science",   "count": 75,  "avg_rate": 44.43, "completed_pct": 57.3},
    ...
  ]
}
```
Compute using SQLAlchemy's `func.count()`, `func.avg()`, `.group_by()`.  
**Round avg_rate to 2 dp, completed_pct to 1 dp.**


---
## 📊 Section 4 — Scoring Rubric

| Task | Points | Key Criteria |
|---|---|---|
| **T1** Database setup (database.py + models.py) | 15 | `get_db()` uses try/finally yield; both ORM classes present; `created_at` uses `func.now()`; `models.Base.metadata.create_all()` called in main.py |
| **T2** auth.py | 10 | `VALID_API_KEYS` dict with 2 keys; `Header(...)` injection; raises HTTP 401 with correct headers; returns role string |
| **T3** CRUD READ endpoints | 20 | `/seed-db` clears + repopulates (seeded=500); `/projects` pagination with skip/limit; `/projects/stats` returns all 7 fields with correct values; `/projects/filter` returns count+data |
| **T4** Background task + prediction log | 15 | `log_to_db` runs via `BackgroundTasks`; response returned before DB write completes; `/predictions/log` endpoint returns log entries |
| **T5** DELETE endpoint | 10 | 404 on missing project_id; 403 on readonly role; 200 + correct JSON on success |
| **T6** test_day148.py | 10 | All 5 assertions pass; 6th bonus test present; pretty-prints output |
| **★ Bonus** `/analytics` endpoint | 10★ | `group_by` category; `avg_rate` correct to 2 dp; `completed_pct` computed from DB counts |
| **Total** | **80 + 10★** | |

### NRA Insight

```
Number:  The /projects/stats endpoint confirms avg hourly rate = ₹43.34/hr
         with 203 high-value projects (>₹50/hr) — 40.6% of the dataset.
Reason:  Over 40% of listed projects exceed ₹50/hr, meaning the API is
         surfacing a premium-segment opportunity that a flat dashboard
         would bury in averages.
Action:  Add a GET /projects/filter?min_rate=50 endpoint in the next
         iteration so the client's app can auto-surface high-value leads;
         set a ₹50/hr floor as the default bid threshold in their workflow.
```

### Interview Framing — Day 148

> *"Why add authentication to an API if it's just an internal tool?"*

> "Authentication is not optional even for internal tools. Without API keys,  
> any service on the same network — or any developer who finds the URL —  
> can query or delete your data. API keys let me issue different keys to different  
> clients, revoke access without redeploying, and audit which key triggered which  
> request in the logs. In Day 148 I implemented two roles: admin (full CRUD)  
> and readonly (GET only). That separation means a client's front-end app  
> can never accidentally trigger a DELETE — it's architecturally impossible  
> with the readonly key."
